<a href="https://colab.research.google.com/github/AnugrahaSaji/AnugrahaSaji/blob/main/PE_quantum_based_sysem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()

Saving 50_voters_with_aadhaar_dataset (4).xlsx to 50_voters_with_aadhaar_dataset (4).xlsx


{'50_voters_with_aadhaar_dataset (4).xlsx': b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xf0$i\\F\xc7MH\x95\x00\x00\x00\xcd\x00\x00\x00\x10\x00\x00\x00docProps/app.xmlM\xcfM\x0b\xc20\x0c\x06\xe0\xbfRv\xb7\x99\x8a\x1e\xa4\x0eD=\x8a\x9e\xbc\xcf.u\x85\xb6)m\x84\xfa\xef\xed\x04?nyy\xc8\x1b\xa2.\x89"&\xb6\x98E\xf1.\xe4m32\xc7\r@\xd6#\xfa>\xcb\xca\xa1\x8a\xa1\xe4{\xae1\xdd\x81\x8c\xb1\x1a\x0f\xa4\x1f\x1e\x03\xc3\xa2m\xd7\x80\x851\x0c8\xcc\xe2\xb7\xb0\xe9\xd4.Fgu\xcf\x96Bw\xb2:Q&\xc3\xe2X4:\xb1\'\x1f\xab\xdc\x1c\n\x10\xe7z%>\x8b\x13K9\x97+\x05\xff\x8bS\xcb\x15S\x9e\xe6\xcao\xfcd\x05\xbf\x07\xba\x17PK\x03\x04\x14\x00\x00\x00\x08\x00\xf0$i\\\xa8l\xa0\xe4\xef\x00\x00\x00+\x02\x00\x00\x11\x00\x00\x00docProps/core.xml\xcd\x92\xc1j\xc30\x0c\x86_e\xf8\x9e\xc8\x89G\xa1&\xf5e\xa5\xa7\r\x06+l\xecfl\xb55\x8bcck$}\xfb%Y\x9b2\xb6\x07\xd8\xd1\xd2\xefO\x9f@\x8d\x89\xd2\x84\x84\xcf)DL\xe40\xdf\r\xbe\xed\xb24q\xc3NDQ\x02dsB\xafs9&\xba\xb1y\x08\xc9k\x1a\x9f\xe9\x08Q\x9b\x0f}D\xa89_\x81G\xd2V\x93\x86\tX\xc4\x85\xc8Tc\x

In [2]:
!pip install qiskit qiskit-aer pycryptodome pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00


In [5]:
# ==========================================
# PERFORMANCE TEST USING EXCEL DATASET
# ==========================================

import pandas as pd
import time
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import hashlib
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

print("\n====== PERFORMANCE TEST WITH EXCEL DATASET ======\n")

# -----------------------------
# LOAD EXCEL FILE
# -----------------------------
file_path = "50_voters_with_aadhaar_dataset (4).xlsx"  # change if filename differs
df = pd.read_excel(file_path)

total_voters = len(df)
print("Total Voters Loaded:", total_voters)

votes_storage = []

total_auth_time = 0
total_encryption_time = 0

overall_start = time.time()

# -----------------------------
# PROCESS EACH VOTER
# -----------------------------
for index, row in df.iterrows():

    start_auth = time.time()

    # -------- QUANTUM KEY --------
    qc = QuantumCircuit(2,2)
    qc.h(0)
    qc.cx(0,1)
    qc.measure([0,1],[0,1])

    sim = Aer.get_backend('aer_simulator')
    qc = transpile(qc, sim)
    result = sim.run(qc, shots=1).result()
    counts = result.get_counts()
    quantum_key = list(counts.keys())[0]

    # -------- HASH KEY --------
    hashed_key = hashlib.sha256(quantum_key.encode()).digest()

    # -------- AES AUTHENTICATION --------
    challenge = get_random_bytes(16)
    cipher = AES.new(hashed_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(challenge)

    cipher2 = AES.new(hashed_key, AES.MODE_GCM, nonce=cipher.nonce)
    response = cipher2.decrypt_and_verify(ciphertext, tag)

    end_auth = time.time()
    total_auth_time += (end_auth - start_auth)

    # -------- VOTE ENCRYPTION --------
    start_enc = time.time()

    vote = random.choice(["A", "B", "C"])
    cipher_vote = AES.new(hashed_key, AES.MODE_GCM)
    encrypted_vote, tag_vote = cipher_vote.encrypt_and_digest(vote.encode())

    votes_storage.append(encrypted_vote.hex())

    end_enc = time.time()
    total_encryption_time += (end_enc - start_enc)

overall_end = time.time()

# -----------------------------
# CALCULATE PERFORMANCE METRICS
# -----------------------------
total_time = overall_end - overall_start
average_auth_time = total_auth_time / total_voters
average_encryption_time = total_encryption_time / total_voters
throughput = total_voters / total_time

# -----------------------------
# DISPLAY RESULTS
# -----------------------------
print("\n====== PERFORMANCE RESULTS ======")
print("Total Voters Processed:", total_voters)
print("Total Execution Time:", round(total_time, 4), "seconds")
print("Average Authentication Time:", round(average_auth_time, 6), "seconds")
print("Average Encryption Time:", round(average_encryption_time, 6), "seconds")
print("System Throughput:", round(throughput, 2), "voters per second")
print("Total Encrypted Votes Stored:", len(votes_storage))


====== PERFORMANCE TEST WITH EXCEL DATASET ======

Total Voters Loaded: 50

====== PERFORMANCE RESULTS ======
Total Voters Processed: 50
Total Execution Time: 9.3525 seconds
Average Authentication Time: 0.186663 seconds
Average Encryption Time: 0.000125 seconds
System Throughput: 5.35 voters per second
Total Encrypted Votes Stored: 50
